# Block-conditioning pipeline debug notebook

This notebook samples four random videos and performs every stage except saving. Model calls and visualizations are intentionally explicit and separated by stage.

In [ ]:
from pathlib import Path
import gc
import random

from IPython.display import Markdown, display
from PIL import Image

from blockconditioning.config import PipelineConfig
from blockconditioning.depth import DA3PointCloudEstimator
from blockconditioning.descriptions import SalientObjectDescriber
from blockconditioning.geometry import compute_object_geometry
from blockconditioning.isolation import KleinObjectIsolator
from blockconditioning.segmentation import Sam3VideoSegmenter
from blockconditioning.video import choose_analysis_indices, decode_and_preprocess_video, list_videos
from blockconditioning.visualization import (
    display_animation,
    display_image_pairs,
    isolation_comparison_images,
    render_3d_boxes_on_black,
    segmentation_overlay_frames,
    side_by_side,
)

# Point this at the dataset root that contains dataset/videos.
DATASET_DIR = Path('/absolute/path/to/dataset')
DEVICE = 'cuda'
RANDOM_SEED = 17

config = PipelineConfig(dataset_dir=DATASET_DIR, device=DEVICE)
analysis_fps = config.video.fps * (
    (config.video.analysis_frame_count - 1) / (config.video.frame_count - 1)
)
config

## Select four random videos

In [ ]:
all_videos = list_videos(config.videos_dir)
if len(all_videos) < 4:
    raise ValueError(f'Expected at least four videos in {config.videos_dir}; found {len(all_videos)}')

selected_videos = random.Random(RANDOM_SEED).sample(all_videos, k=4)
display(Markdown('\n'.join(f'- `{path.name}`' for path in selected_videos)))

## Stage 1 — decode, crop to 97 frames at 16 fps, and resize to 256p

The displayed animations are in-memory JavaScript animations; no debug videos are written.

In [ ]:
processed_by_video = {
    path: decode_and_preprocess_video(path, config.video)
    for path in selected_videos
}

for path, video in processed_by_video.items():
    display(Markdown(f'### {path.name}'))
    display(display_animation(video.frames, video.fps, title=f'{path.name}: 97 frames at 16 fps'))

## Stage 2 — describe up to three salient first-frame objects with OpenAI (luna, low)

In [ ]:
describer = SalientObjectDescriber(config.openai)
descriptions_by_video = {
    path: describer.describe(video.frames[0])
    for path, video in processed_by_video.items()
}

for path, descriptions in descriptions_by_video.items():
    display(Markdown(f'### {path.name}'))
    display(Image.fromarray(processed_by_video[path].frames[0]))
    lines = '\n'.join(f'{index + 1}. {text}' for index, text in enumerate(descriptions))
    display(Markdown(lines or '*No salient objects returned.*'))

## Stage 3 — text-prompted SAM3 video masks on 30 frames

Each prompt keeps only its highest-ranked instance track. The right half uses a distinct color per object.

In [ ]:
analysis_indices = choose_analysis_indices(
    config.video.frame_count,
    config.video.analysis_frame_count,
)
analysis_frames_by_video = {
    path: video.frames[analysis_indices]
    for path, video in processed_by_video.items()
}

segmenter = Sam3VideoSegmenter(config.sam3, device=config.device)
tracks_by_video = {
    path: segmenter.segment(analysis_frames_by_video[path], descriptions_by_video[path])
    for path in selected_videos
}

for path in selected_videos:
    frames = analysis_frames_by_video[path]
    overlays = segmentation_overlay_frames(frames, tracks_by_video[path])
    comparison = side_by_side(frames, overlays)
    display(Markdown(f'### {path.name}: source / SAM3 masks'))
    display(display_animation(comparison, analysis_fps))

del segmenter
gc.collect()

## Stage 4 — DA3 depth, cameras, and video point cloud inputs

DA3 processes all 30 sampled views jointly. The next stage unprojects its depth into the shared world frame.

In [ ]:
depth_estimator = DA3PointCloudEstimator(config.da3, device=config.device)
depth_by_video = {
    path: depth_estimator.predict(analysis_frames_by_video[path])
    for path in selected_videos
}

for path, result in depth_by_video.items():
    display(Markdown(
        f'**{path.name}** — depth `{result.depth.shape}`, intrinsics '
        f'`{result.intrinsics.shape}`, extrinsics `{result.extrinsics_world_to_camera.shape}`'
    ))

del depth_estimator
gc.collect()

## Stage 5 — erode masks, remove 3D outliers, compute and render boxes

The right half is black except for the projected DA3-world axis-aligned boxes.

In [ ]:
geometry_by_video = {
    path: compute_object_geometry(
        tracks_by_video[path],
        depth_by_video[path],
        config.geometry,
    )
    for path in selected_videos
}

for path in selected_videos:
    frames = analysis_frames_by_video[path]
    boxes = render_3d_boxes_on_black(
        geometry_by_video[path],
        depth_by_video[path],
        output_height=frames.shape[1],
        output_width=frames.shape[2],
    )
    comparison = side_by_side(frames, boxes)
    display(Markdown(f'### {path.name}: source / projected 3D boxes'))
    display(display_animation(comparison, analysis_fps))

## Stage 6 — padded first-frame crops and FLUX.2 Klein isolation

For each object, the 512×512 input crop is shown on the left and the square 256×256 isolated result on the right.

In [ ]:
isolator = KleinObjectIsolator(config.klein, device=config.device)
isolated_by_video = {
    path: isolator.isolate(
        processed_by_video[path].frames[0],
        geometry_by_video[path],
    )
    for path in selected_videos
}

for path in selected_videos:
    display(Markdown(f'### {path.name}'))
    figure = display_image_pairs(
        isolation_comparison_images(isolated_by_video[path])
    )
    if figure is not None:
        display(figure)

del isolator
gc.collect()

There is intentionally no saving cell. Use `VideoPipeline.process_and_save` or the `blockconditioning` CLI only after the debug outputs are satisfactory.